In [1]:
import NNSA

# List of available models:

In [2]:
NNSA.available_models()

BaSTIModel (http://basti-iac.oa-abruzzo.inaf.it/)
PARSECModel (https://stev.oapd.inaf.it/PARSEC/)
MISTModel (https://waps.cfa.harvard.edu/MIST/)
SYCLISTModel (https://www.unige.ch/sciences/astro/evolution/en/database/syclist)
DartmouthModel (https://rcweb.dartmouth.edu/stellar/)
YaPSIModel (http://www.astro.yale.edu/yapsi/)


# Load BaSTI model:
#### By default, the age models use sklearn to run the neural networks. If you do not have it installed, or don't want to use it, you can pass the argument 'use_sklearn=False' to switch to a numpy version. Be aware though, in this mode it becomes significantly slower once you run the models on more than ~100 stars.

In [4]:
age_model = NNSA.BaSTIModel()
#age_model = NNSA.BaSTIModel(use_sklearn=False)

# Estimate the age of a single star using [M/H],MG,BPRP:
#### To estimate the age of a star, use the ages_prediction() method, and pass it a metallicity MH, magnitude MG and color BPRP. Be aware with only these values provided, the return value is a np.array of shape (1,1). The reason will be clear if you check later uses. 

In [10]:
age_model.ages_prediction(MH=0.0,MG=4.0,BPRP=1.0)

array([[13.89808636]])

#### You can use the check_domain() method to verify that the star is within the bounds of the isochrones points used for training the neural networks. The function only checks for this in the HR Diagram, so you pass it a metallicity MH, magnitude MG and color BPRP.

In [8]:
age_model.check_domain(MH=0.0,MG=4.0,BPRP=1.0)

array([ True])

# Estimate the age of a single star using [M/H],MG,BPRP,GBP & GRP:
#### You can also pass the red GRP and blue GBP magnitudes separately as additional inputs. This generally results in better estimations.

In [14]:
age_model.ages_prediction(MH=0.0,MG=2.0,BPRP=1.0,GBP=2.0,GRP=1.0)

array([[1.38048182]])

# Estimate the age of multiple stars:
#### If you pass lists of values of size N as inputs instead of single values, the return value of the ages_prediction() method will be a np.array of shape (N,1). Again, the reason for the 2D shape will be clear by checking later uses.

In [ ]:
age_model.ages_prediction(MH=[0.0,-1.0],MG=[2.0,3.0],BPRP=[1.0,0.5])

array([[1.6267248 ],
       [5.99103612]])

In [16]:
age_model.ages_prediction(MH=[0.0,-1.0],MG=[2.0,3.0],BPRP=[1.0,0.5],GBP=[2.0,1.0],GRP=[1.0,0.5])

array([[1.38048182],
       [2.16979408]])

#### You can also check if the stars are within the training domain all at once by passing lists of metallicity MH, magnitude MG & color values BPRP.

In [7]:
age_model.check_domain(MH=[0.0,-1.0],MG=[2.0,3.0],BPRP=[1.0,0.5])

array([ True,  True])

#### That way, you can crossmatch both outputs to only keep stars whose age estimates we can reliably trust:

In [16]:
MH = [0.0,-1.0,-2.0,-3.0]
MG = [2.0,3.0,3.0,3.0]
BPRP = [1.0,0.5,0.5,0.5]
GBP = [2.0,1.0,1.0,1.0]
GRP = [1.0,0.5,0.5,0.5]
ages = age_model.ages_prediction(MH=MH,MG=MG,BPRP=BPRP,GBP=GBP,GRP=GRP)
print(ages)
outside_domain = age_model.check_domain(MH=MH,MG=MG,BPRP=BPRP)
print(outside_domain)
ages = ages[outside_domain]
print(ages)

[[1.38048182]
 [2.16979408]
 [2.88745321]
 [3.4155044 ]]
[ True  True  True False]
[[1.38048182]
 [2.16979408]
 [2.88745321]]


# Estimate the age of a single star with n=10 Monte Carlo realisations using uncertainties:
#### If you have uncertainty values for a star, you can pass them to the ages_prediction() method by adding values for eMH, eMG & eBPRP. The idea is that multiple age predictions will be made by offsetting the star's metallicity MH, magnitude MG & color BPRP by adding a random gaussian offset of width (eMH,eMG,eBPRP). The number of realisations made is controlled by the parameter n. The output of the method will then be a np.array of shape (1,n).

In [9]:
age_model.ages_prediction(MH=0.0,MG=2.0,BPRP=1.0,eMH=0.1,eMG=0.1,eBPRP=0.1,n=10)

array([[ 2.1245525 ,  1.73907248,  8.16107031,  1.88776774,  7.58729075,
         7.01121024, 13.42280508,  2.80697195, 12.10321434,  2.18367215]])

#### Once the ages have been predicted for each star n times, you can call statistical methods model.mean_ages(), model.median_ages(), model.mode_ages() and model.std_ages()

In [10]:
print(age_model.mean_ages())
print(age_model.median_ages())
print(age_model.mode_ages())
print(age_model.std_ages())

[5.90276275]
[4.9090911]
[2.15]
[4.18700143]


# Estimate the age of multiple stars with n=10 Monte Carlo realisations using uncertainties:
#### Finally, you can combine multiple (N) stars age estimations and added uncertainties, with n Monte Carlo realisations. The output of the ages_prediction() method will then be a np.array of shape (N,n). A value of n=100 seems to be a good middle ground between speed and accurate predictions. If the number of stars is low, n>=1000 gives enough realisations to draw a PDF.

In [7]:
age_model.ages_prediction(MH=[0.0,-1.0],MG=[2.0,3.0],BPRP=[1.0,0.5],eMH=[0.1,0.1],eMG=[0.1,0.1],eBPRP=[0.1,0.1],n=10)

array([[ 1.74337549,  2.24078317, 13.85054882,  2.18587654,  1.62326973,
         1.60462393,  1.74503609,  1.46790738,  1.68988403,  2.48510483],
       [ 2.44813284,  5.73527402,  5.46770581,  5.44741403,  6.64279109,
         6.1754457 ,  4.55708287,  4.84676545,  6.66956471,  6.62737005]])

In [8]:
print(age_model.mean_ages())
print(age_model.median_ages())
print(age_model.mode_ages())
print(age_model.std_ages())

[3.063641   5.46175466]
[1.74420579 5.60148992]
[1.65 6.65]
[3.60913347 1.2278143 ]
